<a href="https://colab.research.google.com/github/yuli894/DL_learning/blob/main/Transformer_test1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install matplotlib

In [1]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# 设置随机种子
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # 添加 CUDA 随机种子设置

# 超参数设置（优化后）
VOCAB_SIZE = 10
EMBED_DIM = 128
NUM_HEADS = 4
HIDDEN_DIM = 64
NUM_ENCODER_LAYERS = 2
MAX_SEQ_LEN = 20
BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

# =======================
# 1. 合成数据集定义
# =======================
class SyntheticTextDataset(Dataset):
    def __init__(self, num_samples):
        self.data = []
        self.labels = []
        for _ in range(num_samples):
            seq_len = random.randint(5, MAX_SEQ_LEN)
            tokens = np.random.randint(1, VOCAB_SIZE, size=seq_len).tolist()
            threshold = seq_len * VOCAB_SIZE / 2
            label = 1 if sum(tokens) > threshold else 0  # 模型学到的内容
            self.data.append(tokens)
            self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

# pad对齐序列；mask标明填充标记，防止误读
def collate_fn(batch):
    data, labels = zip(*batch)
    lengths = [len(seq) for seq in data]
    max_len = max(lengths)
    padded = [F.pad(seq, (0, max_len - len(seq)), value=0) for seq in data]  # 优化填充方式
    padded_seqs = torch.stack(padded)
    mask = (padded_seqs == 0)
    return padded_seqs, mask, torch.stack(labels)


train_dataset = SyntheticTextDataset(1000)
test_dataset = SyntheticTextDataset(200)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# =======================
# 2. 位置编码模块
# =======================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # [max_len, d_model]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # [max_len, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  # [d_model//2]

        pe[:, 0::2] = torch.sin(position * div_term)  # [max_len, d_model//2]
        pe[:, 1::2] = torch.cos(position * div_term)  # [max_len, d_model//2]

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return x




# =======================
# 3. Transformer分类器
# =======================
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim, max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, src, src_key_padding_mask):
        x = self.embedding(src)  # (batch, seq, embed_dim)
        x = self.pos_encoder(x)
        x = x.transpose(0, 1)  # (seq, batch, embed_dim)
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        x = x.mean(dim=0)  # ⭐ 优化：平均池化（比 x[0,:,:] 更稳定）
        return self.fc(x)

# 初始化模型与设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_ENCODER_LAYERS,
    num_classes=2,
    max_seq_len=MAX_SEQ_LEN
).to(device)
print("是否使用GPU:", next(model.parameters()).is_cuda)

# =======================
# 4. 训练与评估函数
# =======================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for src, mask, labels in loader:
        src, mask, labels = src.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(src, src_key_padding_mask=mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * src.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += src.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for src, mask, labels in loader:
            src, mask, labels = src.to(device), mask.to(device), labels.to(device)
            outputs = model(src, src_key_padding_mask=mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * src.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += src.size(0)
    return total_loss / total, correct / total

# =======================
# 5. 训练主循环
# =======================
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
          f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}")

# =======================
# 6. 模型预测示例
# =======================
model.eval()
sample_src, sample_mask, sample_label = next(iter(test_loader))
sample_src, sample_mask = sample_src.to(device), sample_mask.to(device)
with torch.no_grad():
    output = model(sample_src, src_key_padding_mask=sample_mask)
    pred = output.argmax(1)
print("样本真实标签：", sample_label.numpy())
print("模型预测标签：", pred.cpu().numpy())


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


是否使用GPU: False
Epoch 1: Train Loss=0.4170, Train Acc=0.7960, Test Loss=0.2169, Test Acc=0.9000
Epoch 2: Train Loss=0.1882, Train Acc=0.9220, Test Loss=0.1671, Test Acc=0.9350
Epoch 3: Train Loss=0.1457, Train Acc=0.9430, Test Loss=0.1081, Test Acc=0.9500
Epoch 4: Train Loss=0.1597, Train Acc=0.9410, Test Loss=0.0925, Test Acc=0.9600
Epoch 5: Train Loss=0.1871, Train Acc=0.9180, Test Loss=0.0994, Test Acc=0.9750
Epoch 6: Train Loss=0.1111, Train Acc=0.9600, Test Loss=0.0708, Test Acc=0.9800
Epoch 7: Train Loss=0.0730, Train Acc=0.9690, Test Loss=0.2155, Test Acc=0.9300
Epoch 8: Train Loss=0.1552, Train Acc=0.9300, Test Loss=0.0724, Test Acc=0.9700
Epoch 9: Train Loss=0.0735, Train Acc=0.9730, Test Loss=0.0764, Test Acc=0.9750
Epoch 10: Train Loss=0.0656, Train Acc=0.9730, Test Loss=0.0650, Test Acc=0.9800
Epoch 11: Train Loss=0.0868, Train Acc=0.9620, Test Loss=0.0580, Test Acc=0.9800
Epoch 12: Train Loss=0.0994, Train Acc=0.9560, Test Loss=0.1007, Test Acc=0.9650
Epoch 13: Train Loss=0